In [ ]:
pip install requests dotenv twilio -q

In [ ]:
import requests
import os
from pathlib import Path
from dotenv import load_dotenv
import json

load_dotenv()
ACCOUNT_SID = os.environ["TWILIO_ACCOUNT_SID"]
AUTH_TOKEN  = os.environ["TWILIO_AUTH_TOKEN"]
AUTH = (ACCOUNT_SID, AUTH_TOKEN)

import os
from twilio.rest import Client

client = Client(ACCOUNT_SID, AUTH_TOKEN)

MEMORY_BASE  = "https://memory.twilio.com/v1"
CONV_BASE    = "https://conversations.twilio.com/v2"

Channel

In [ ]:

url = "https://conversations.twilio.com/v1/Services/IS37231a3da5824da58229cd7760041bce/Conversations"

response = requests.post(
    url,
    auth=(ACCOUNT_SID, AUTH_TOKEN)
)
# response.json()
print(response.json()['sid'])
channelSID = response.json()['sid']

In [ ]:
memory_store = "mem_store_01m11g22dffbaawr3d2vx99mfb"
orchestrator_config = "conv_configuration_01m28q62mwfe398a6pz3w1hp2b"

In [ ]:
profile_A = "mem_profile_01m28hadgwfaqbjqhdnzjektts"
profile_B = "mem_profile_01m28ha8cjfvq94rjkbgfdd522"
AI_AGENT_profile = "mem_profile_01m2de4b06egxbwteesrx912nr"
AGENT_NUMBER = "+447575583935"

In [ ]:
def create_conversation(configuration_id, customer_profile, agent_profile, channel_id):
    payload = {
        "configurationId": configuration_id,
        "participants": [
            {
                "type": "CUSTOMER",
                "profileId": customer_profile,
                "addresses": [{ "address":"+447876762080",
                                                        "channel":"SMS",
                                                        "channelId":channel_id
                                                        }]
            },
            {
                "type": "AI_AGENT",
                "profileId": agent_profile,
                "addresses": [{ "address":"+447575583935",
                                        "channel":"SMS",
                                        "channelId":channel_id
                                        }]
            },
        ],
    }
    comm_resp = requests.post(
            f"{CONV_BASE}/Conversations",
            auth=AUTH,
            headers={"Content-Type": "application/json"},
            json=payload,
        )
    return comm_resp

In [ ]:
conversation_config_A = create_conversation(orchestrator_config, profile_A, AI_AGENT_profile, channelSID)
conversation_config_A.json()

In [ ]:
conv_A_id = conversation_config_A.json()["id"]
print(f"Conversation A ID: {conv_A_id}")
A_customer_participant = conversation_config_A.json()['participants'][0]
print(f"Customer Participant in Conversation A: {A_customer_participant}")
A_agent_participant = conversation_config_A.json()['participants'][1]

In [ ]:
print(A_customer_participant['addresses'][0]['address'])
print(A_customer_participant['id'])
print(A_agent_participant['addresses'][0]['address'])
print(A_agent_participant['id'])

In [ ]:
url = "https://conversations.twilio.com/v1/Services/IS37231a3da5824da58229cd7760041bce/Conversations"

response = requests.post(
    url,
    auth=(ACCOUNT_SID, AUTH_TOKEN)
)
# response.json()
print(response.json()['sid'])
channelSID = response.json()['sid']

In [ ]:
conversation_config_B = create_conversation(orchestrator_config, profile_B, AI_AGENT_profile, channelSID)
conversation_config_B.json()

In [ ]:
conv_B_id = conversation_config_B.json()["id"]
print(f"Conversation B ID: {conv_B_id}")
B_customer_participant = conversation_config_B.json()['participants'][0]
print(f"Customer Participant in Conversation B: {B_customer_participant}")
B_agent_participant = conversation_config_B.json()['participants'][1]

print(B_customer_participant['addresses'][0]['address'])
print(B_customer_participant['id'])
print(B_agent_participant['addresses'][0]['address'])
print(B_agent_participant['id'])

In [ ]:
def send_message(conversation_id, sender_id, recipient_id, sender_number, recipient_number, body):
    payload = {
        "type": "SEND_MESSAGE",
        "payload": {
            "to": [{"channel": "SMS", "participantId": recipient_id, "address": recipient_number}],
            "from": {"channel": "SMS", "participantId": sender_id, "address": sender_number},
            "content": {"text": body},
        },
    }
    message_resp = requests.post(
        f"{CONV_BASE}/Conversations/{conversation_id}/Actions",
        auth=AUTH,
        headers={"Content-Type": "application/json"},
        json=payload,
    )
    return message_resp

In [ ]:
res_A = send_message(conv_A_id, A_agent_participant['id'], A_customer_participant['id'], AGENT_NUMBER, "+447876762080", "Disco mambo is your first song")
res_A.json()

In [ ]:
res_A_tech = send_message(conv_A_id, A_customer_participant['id'], A_agent_participant['id'], "+447876762080", AGENT_NUMBER, "Yes I am . problem with the heater. i can fix in two hours.")
res_A_tech.json()

In [ ]:
res_B_agent = send_message(conv_B_id, B_agent_participant['id'], B_customer_participant['id'], AGENT_NUMBER, "+447876762080", "Are you still loking for the house")
res_B_agent.json()

In [ ]:
res_B_tech = send_message(conv_B_id, B_customer_participant['id'], B_agent_participant['id'], "+447876762080", AGENT_NUMBER, "You sent me the wrong address")
res_B_tech.json()

In [ ]:
def send_comms(conversation_id, sender_id, recipient_id, sender_number, recipient_number, channelSID, body):
    payload = {
        "author": {
            "address": sender_number,
            "channel": "SMS",
            "participantId": sender_id
        },
        "content": {
            "type": "TEXT",
            "text": body,
        },
        "channelId": channelSID,
        "recipients": [
            {
            "address": recipient_number,
            "channel": "SMS",
            "participantId": recipient_id
            }
        ]
        }
    message_resp = requests.post(
        f"{CONV_BASE}/Conversations/{conversation_id}/Communications",
        auth=AUTH,
        headers={"Content-Type": "application/json"},
        json=payload,
    )
    return message_resp

In [ ]:
res_B = send_comms(conv_B_id, B_agent_participant['id'], B_customer_participant['id'], AGENT_NUMBER, "+447876762080", "CH932b223b0a1e4867bb88716599fac7bb", "I sent it to you already. did you get it" )
res_B.json()

In [ ]:
res_A = send_comms(conv_A, "conv_participant_01m2dpy4seenjty6e7vsjmkzme", "conv_participant_01m2dpy4see2dbpmxsze0jv9zh", AGENT_NUMBER, "+447876762080", "how is the job? what is wrong?")
res_A.json()

In [ ]:
res_B = send_comms(conv_B, "conv_participant_01m2dq7nz9fzws5v0x7g4ec173", "conv_participant_01m2dq7nz9ffeaj0g71t4wqhyw", "+447876762080", AGENT_NUMBER, "Boiler isn't working. its a 2 day job. I will havew to leave soon")
res_B.json()

In [ ]:
res_A = send_comms(conv_A, "conv_participant_01m2dpy4see2dbpmxsze0jv9zh", "conv_participant_01m2dpy4seenjty6e7vsjmkzme", "+447876762080", AGENT_NUMBER, "computer needs to bne replaced. two hour job")
res_A.json()